In [1]:
%pip install python-dotenv openai datasets math_verify tqdm torch aiolimiter

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
import logging

logging.basicConfig(level=logging.INFO)

In [4]:
import os
from openai import AsyncOpenAI
from aiolimiter import AsyncLimiter
from asyncio import Semaphore
from math_verify import parse
from dataclasses import dataclass

NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")
MODEL_ID = "openai/gpt-oss-20b"

client = AsyncOpenAI(
	base_url="https://integrate.api.nvidia.com/v1",
	api_key=NVIDIA_API_KEY,
	max_retries=0,
)

@dataclass
class MyCompletionChoice:
	reasoning_content: str = ""
	content: str = ""

limiter = AsyncLimiter(40)
semaphore = Semaphore(10)
async def create_completion(*args, **kwargs):
	kwargs["stream"] = True
	while True:
		try:
			async with limiter:
				async with semaphore:
					choices: list[MyCompletionChoice] = []
					async for chunk in await client.chat.completions.create(*args, **kwargs):
						for choice in chunk.choices:
							while len(choices) <= choice.index:
								choices.append(MyCompletionChoice())

							if delta := getattr(choice.delta, "reasoning_content", None):
								choices[choice.index].reasoning_content += delta

							if delta := getattr(choice.delta, "content", None):
								choices[choice.index].content += delta
					return choices
		except:
			pass

prompt = "What is 13 times 17? Box your answer."
gold = "221"

choices = await create_completion(
	model=MODEL_ID,
	messages=[{"role": "user", "content": prompt}],
	max_tokens=2**10,
    n=8,
	extra_body={
        "chat_template_kwargs": {"enable_thinking": True},
        "reasoning_budget": 2**10
    },
)

print("*"*20, "Prompt", "*"*20)
print(prompt)

parsed_gold = parse(gold) or [None]
for i, choice in enumerate(choices):
  parsed_answer = parse(choice.content) or [None]
  correct = parsed_gold[0] == parsed_answer[0]

  print("*"*20, f"Choice {i+1}: {parsed_answer[0]} ({'correct' if correct else 'incorrect'})", "*"*20)
  if correct:
    print(f"<think>{choice.reasoning_content}</think>{choice.content}")

INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"


******************** Prompt ********************
What is 13 times 17? Box your answer.
******************** Choice 1: 221 (correct) ********************
<think>The question: "What is 13 times 17? Box your answer." So answer is 221. They want 13*17=221.

They want to "box" our answer. So likely respond with a box, maybe using ASCII or LaTeX? Probably bracket and underscores? Many ways. But the instructions: "Box your answer." So let's show a box: 
```
+-----+
| 221 |
+-----+
```
Or use box drawing characters:

```
┌─────┐
│ 221 │
└─────┘
```

We need to answer with 221 boxed. Provide the answer in the final.</think>Here’s the boxed answer:

```
┌─────┐
│ 221 │
└─────┘
```
******************** Choice 2: 221 (correct) ********************
<think>We need to answer 13*17. 13*17 = 221. They also want the answer "boxed". So maybe output a box around 221. Use formatting. For example:

```
+-----+
| 221 |
+-----+
```

Or maybe a single box with the number: (221) or something. We'll include a te

In [5]:
from datasets import load_dataset

ds = load_dataset("open-r1/OpenR1-Math-220k", "default")
ds

INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/open-r1/OpenR1-Math-220k/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/open-r1/OpenR1-Math-220k/e4e141ec9dea9f8326f4d347be56105859b2bd68/README.md "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/open-r1/OpenR1-Math-220k/resolve/e4e141ec9dea9f8326f4d347be56105859b2bd68/OpenR1-Math-220k.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/open-r1/OpenR1-Math-220k/open-r1/OpenR1-Math-220k.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/open-r1/OpenR1-Math-220k/revision/e4e141ec9dea9f8326f4d347be56105859b2bd68 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/open-r1/OpenR1-Math-220k/resolve/e4e141ec9dea9f8326f4d347be56105859b2bd68/.huggingface.yaml "HTTP/1.1 404 

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/open-r1/OpenR1-Math-220k/resolve/e4e141ec9dea9f8326f4d347be56105859b2bd68/dataset_infos.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/open-r1/OpenR1-Math-220k/tree/e4e141ec9dea9f8326f4d347be56105859b2bd68/data?recursive=true&expand=false "HTTP/1.1 200 OK"


DatasetDict({
    train: Dataset({
        features: ['problem', 'solution', 'answer', 'problem_type', 'question_type', 'source', 'uuid', 'is_reasoning_complete', 'generations', 'correctness_math_verify', 'correctness_llama', 'finish_reasons', 'correctness_count', 'messages'],
        num_rows: 93733
    })
})

In [6]:
from tqdm.contrib.logging import logging_redirect_tqdm
from tqdm.asyncio import tqdm_asyncio
from math_verify import verify
import torch
from datasets import Dataset

async def generate_dataset(prompts, golds, **kwargs):
	dataset_dict = {
		"prompt": [],
		"outputs": [],
		"advantages": [],
	}
	futures = []
	for prompt in prompts:
		futures.append(create_completion(
			**kwargs,
			messages=[{"role": "user", "content": prompt}],
		))
	with logging_redirect_tqdm():
		completions = await tqdm_asyncio.gather(*futures, desc="Creating completions")
	for prompt, choices, gold in zip(prompts, completions, golds):
		gold = parse(gold)

		outputs = []
		rewards = []
		for choice in choices:
			answer = parse(choice.content)
			correct = verify(gold, answer)

			outputs.append(f"<think>{choice.reasoning_content}</think>{choice.content}")
			rewards.append(1.0 if correct else 0.0)
		rewards = torch.tensor(rewards, dtype=torch.float32)

		rewards_std = rewards.std()
		if rewards_std < 1e-5:
			advantages = torch.zeros_like(rewards)
		else:
			advantages = (rewards - rewards.mean()) / rewards_std

		dataset_dict["prompt"].append(prompt)
		dataset_dict["outputs"].append(outputs)
		dataset_dict["advantages"].append(advantages.tolist())
	return Dataset.from_dict(dataset_dict)

example_ds = await generate_dataset(
	prompts=[
		"Pick an random integer from 1 to 3. Don't pick 2. Box your answer.",
		"What is 8 times 3? Box your answer.",
	],
	golds=["3", "24"],
	model=MODEL_ID,
	max_tokens=2**10,
    n=8,
	extra_body={
        "chat_template_kwargs": {"enable_thinking": True},
        "reasoning_budget": 2**10,
    },
)
example_ds[:]

INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
Creating completions: 100%|██████████| 2/2 [00:02<00:00,  1.48s/it]


{'prompt': ["Pick an random integer from 1 to 3. Don't pick 2. Box your answer.",
  'What is 8 times 3? Box your answer.'],
 'outputs': [['<think>User: "Pick a random integer from 1 to 3. Don\'t pick 2. Box your answer." They want a random integer but not 2. So options: 1 or 3. They want answer boxed. Likely they want a single number in a box. I can choose randomly between 1 and 3. Let\'s decide: maybe 3. But the prompt "Pick a random integer from 1 to 3. Don\'t pick 2." So we choose 1 or 3. I\'ll randomly choose 1. But maybe I\'d choose 3. It doesn\'t matter. But they said "randomly" but we can\'t truly random. But it\'s a simple choice. Let\'s pick 3. We\'ll box answer. Use markdown curly braces? Box can be achieved with \\boxed{3} or similar. Let\'s use \\boxed{3} or with a box: [3]. Use latex: $\\boxed{3}$. That should be fine.</think>\\[\n\\boxed{3}\n\\]',
   '<think>The user asks: "Pick an random integer from 1 to 3. Don\'t pick 2. Box your answer." Essentially they want a random

In [7]:
input_ds = ds["train"].shuffle().select(range(128))
output_ds = await generate_dataset(
	prompts=input_ds["problem"],
	golds=input_ds["answer"],
	model=MODEL_ID,
	max_tokens=2**15,
    n=64,
	extra_body={
        "chat_template_kwargs": {"enable_thinking": True},
        "reasoning_budget": 2**15,
    },
)
output_ds

INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completion

Dataset({
    features: ['prompt', 'outputs', 'advantages'],
    num_rows: 128
})

In [8]:
output_ds.to_parquet("reg_grpo_128.parquet")

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

100403699